# BA Thesis – Phase 1: Training auf SmallMinesDS (Kaggle)

**Ziel:** Prithvi-EO v2 (300M) auf dem SmallMinesDS-Datensatz trainieren und einen validen Checkpoint erzeugen, mit dem auf Basis der Trainingsdaten korrekte Galamsey-Vorhersagen möglich sind.

**Modell:** Prithvi-EO v2 (300M) + UperNet-Decoder  
**Daten:** SmallMinesDS – 6 Bänder: B2, B3, B4, B8A, B11, B12 (Band-Fix v2)  
**Backbone:** eingefroren (nur Decoder/Head wird trainiert)

---

## Kaggle-Setup (einmalig)

1. **Dataset hochladen:** `data/GhanaMiningPrithvi/` lokal als ZIP verpacken und auf Kaggle als Dataset `ghana-mining-prithvi` hochladen:
   ```bash
   zip -r GhanaMiningPrithvi.zip data/GhanaMiningPrithvi/
   ```
2. Dieses Notebook auf [kaggle.com/code](https://www.kaggle.com/code) hochladen → **Add data** → `ghana-mining-prithvi` hinzufügen
3. **GPU aktivieren:** Settings → Accelerator → **GPU P100** (empfohlen) oder T4
4. Zellen von oben nach unten ausführen
5. Checkpoint nach Abschluss über das **Output-Panel** herunterladen → lokal als `models/prithvi-v2-300-base.ckpt` speichern

## Zelle 1: Pakete installieren

In [ ]:
import subprocess, sys

# ── Schritt 1: Kaggle's exakte numpy-Version ermitteln ─────────────────────────
# Kaggle hat numpy + rasterio + torch etc. vorinstalliert, alle C-Extensions sind
# für GENAU diese numpy-Version kompiliert. Ändert pip numpy auch nur um eine
# Minor-Version (z.B. 2.2.3 → 2.0.2), brechen alle C-Extensions → ImportError.
_np_ver = subprocess.check_output(
    [sys.executable, '-c', 'import numpy; print(numpy.__version__)']
).decode().strip()
print(f"Kaggle numpy (darf sich NICHT ändern): {_np_ver}")

# ── Schritt 2: terratorch installieren, numpy exakt pinnen ────────────────────
# numpy=={_np_ver} verhindert, dass pip numpy auf irgendeine andere Version ändert.
# Sollte terratorch eine inkompatible numpy-Version erzwingen wollen, schlägt pip
# mit einem klaren Fehler fehl (statt mysteriösem Runtime-Crash).
!pip install -q terratorch==0.99.7 numpy=={_np_ver}

# ── Schritt 3: Verifizieren ──────────────────────────────────────────────────
_np_after = subprocess.check_output(
    [sys.executable, '-c', 'import numpy; print(numpy.__version__)']
).decode().strip()
if _np_after != _np_ver:
    print(f"numpy wurde von {_np_ver} auf {_np_after} geaendert!")
    print("   -> Bitte 'Run > Restart Session', dann ab Zelle 2 weiter")
else:
    print(f"numpy {_np_ver} unveraendert – weiter mit Zelle 2 (kein Restart noetig)")

## Zelle 2: Imports & Pfade

In [ ]:
import os
import numpy as np
import torch
import rasterio
import albumentations as A
from albumentations.pytorch import ToTensorV2

from terratorch.models import PrithviModelFactory
from terratorch.datamodules import GenericNonGeoSegmentationDataModule
from terratorch.tasks import SemanticSegmentationTask
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, RichProgressBar
from lightning.pytorch.loggers import TensorBoardLogger

# ── Pfade ──────────────────────────────────────────────────────────────────────
CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# Automatische Pfad-Suche: findet GhanaMiningPrithvi/ unabhängig von der Kaggle-Mounting-Tiefe
import subprocess
result = subprocess.run(
    ['find', '/kaggle/input', '-type', 'd', '-name', 'GhanaMiningPrithvi'],
    capture_output=True, text=True
)
found_paths = [p.strip() for p in result.stdout.strip().split('\n') if p.strip()]

if found_paths:
    DATASET_INPUT = found_paths[0]
    print(f"Dataset gefunden: {DATASET_INPUT}")
else:
    # Fallback auf letzten bekannten Pfad
    DATASET_INPUT = '/kaggle/input/datasets/mathiashaberger/ghanaminingprithvi-ashanti-smds-dataset/data/GhanaMiningPrithvi'
    print(f"WARNUNG: Auto-Suche fehlgeschlagen. Fallback: {DATASET_INPUT}")
    print("Verfügbare Inhalte in /kaggle/input/:")
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        if level < 4:
            print('  ' + '  ' * level + os.path.basename(root) + '/')
        if level >= 4:
            dirs.clear()

train_dir = os.path.join(DATASET_INPUT, 'training')
val_dir   = os.path.join(DATASET_INPUT, 'validation')

# ── Daten prüfen ───────────────────────────────────────────────────────────────
if not os.path.isdir(train_dir):
    raise FileNotFoundError(f"training/ nicht gefunden: {train_dir} – Dataset korrekt hinzugefügt?")

n_train = len([f for f in os.listdir(train_dir) if f.endswith('_IMG.tif')])
n_val   = len([f for f in os.listdir(val_dir)   if f.endswith('_IMG.tif')])
print(f"Training:   {n_train} Patches")
print(f"Validation: {n_val} Patches")
if n_train == 0 or n_val == 0:
    raise ValueError("training/ oder validation/ ist leer.")

# Band-Check: jede Datei muss genau 6 Bänder haben (Band-Fix v2)
sample = [f for f in os.listdir(train_dir) if f.endswith('_IMG.tif')][0]
with rasterio.open(os.path.join(train_dir, sample)) as s:
    n_bands = s.count
print(f"Band-Check: '{sample}' hat {n_bands} Bänder (erwartet: 6)")
if n_bands != 6:
    raise ValueError("Patch hat nicht 6 Bänder! 01_prepare_dataset.py (v2) lokal ausführen und neu hochladen.")

print(f"\nGPU verfügbar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Zelle 3: Konfiguration

In [ ]:
# ── Normalisierung ──────────────────────────────────────────────────────────────
# Statistiken für B2, B3, B4, B8A, B11, B12 auf SmallMinesDS
MEANS = [1473.81388377, 1703.35249650, 1696.67685941, 3832.39764247, 3156.11122121, 2226.06822112]
STDS  = [ 223.43533204,  285.53613398,  413.82320306,  389.61483882,  451.49534791,  468.26765909]
BANDS = ["BLUE", "GREEN", "RED", "VNIR_5", "SWIR_1", "SWIR_2"]

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    ToTensorV2()
])

datamodule = GenericNonGeoSegmentationDataModule(
    batch_size=8,
    num_workers=4,
    train_data_root=train_dir,
    val_data_root=val_dir,
    test_data_root=val_dir,
    img_grep='*_IMG.tif',
    label_grep='*_MASK.tif',
    means=MEANS,
    stds=STDS,
    num_classes=2,
    train_transform=train_transform,
    dataset_bands=BANDS,
    output_bands=BANDS,
    no_data_replace=0,
    no_label_replace=-1,
)

task = SemanticSegmentationTask(
    model_args={
        "backbone":              "prithvi_eo_v2_300",
        "bands":                 BANDS,
        "in_channels":           6,
        "num_classes":           2,
        "pretrained":            True,   # lädt Prithvi-Gewichte von HuggingFace
        "decoder":               "UperNetDecoder",
        "rescale":               True,
        "backbone_num_frames":   1,
        "head_dropout":          0.1,
        "decoder_scale_modules": True,
    },
    model_factory="PrithviModelFactory",
    loss="ce",
    lr=1e-3,
    ignore_index=-1,
    optimizer="AdamW",
    optimizer_hparams={"weight_decay": 0.05},
    freeze_backbone=True,   # nur Decoder/Head wird trainiert
    class_names=['Non_mining', 'Mining'],
)

datamodule.setup('fit')

n_trainable = sum(p.numel() for p in task.parameters() if p.requires_grad)
n_total     = sum(p.numel() for p in task.parameters())
print(f"Trainierbare Parameter: {n_trainable:,} / {n_total:,} ({n_trainable/n_total*100:.1f}%)")
print("Konfiguration fertig.")

## Zelle 4: Training

**Erwartete Laufzeit:** ~2–3 h auf P100, ~3–5 h auf T4  
Checkpoints werden automatisch in `/kaggle/working/checkpoints/` gespeichert.

In [ ]:
checkpoint_cb = ModelCheckpoint(
    dirpath=CKPT_DIR,
    monitor=task.monitor,
    save_top_k=2,
    save_last=True,
    filename='prithvi-v2-300-epoch{epoch:02d}-val{val_loss:.4f}',
)
early_stop = EarlyStopping(
    monitor=task.monitor,
    min_delta=0.001,
    patience=10,
    verbose=True,
)

trainer = Trainer(
    devices=1,
    precision='16-mixed',
    callbacks=[RichProgressBar(), checkpoint_cb, early_stop],
    logger=TensorBoardLogger(CKPT_DIR, name='logs'),
    max_epochs=50,
    default_root_dir=CKPT_DIR,
    log_every_n_steps=5,
    check_val_every_n_epoch=1,
)

print("Training startet...")
print(f"Checkpoints: {CKPT_DIR}")
print()
trainer.fit(model=task, datamodule=datamodule)

## Zelle 5: Evaluation & Ergebnis

In [ ]:
print("Auswertung auf Validation-Set...")
results = trainer.test(model=task, datamodule=datamodule)

best_ckpt = checkpoint_cb.best_model_path
last_ckpt = checkpoint_cb.last_model_path

print("\n" + "=" * 60)
print("ERGEBNIS")
print("=" * 60)
print(f"Test-Metriken: {results}")
print(f"\nBester Checkpoint: {best_ckpt}")
print(f"Letzter Checkpoint: {last_ckpt}")

print("\n" + "=" * 60)
print("ALLE CHECKPOINTS")
print("=" * 60)
for f in sorted(os.listdir(CKPT_DIR)):
    if f.endswith('.ckpt'):
        size = os.path.getsize(os.path.join(CKPT_DIR, f)) / 1e6
        print(f"  {f}  ({size:.0f} MB)")

print("\n→ Download: rechts im Kaggle-Panel 'Output' öffnen → checkpoints/")
print("  Lokal speichern als: models/prithvi-v2-300-base.ckpt")